# Synthetic M1 → M2
All inputs are fabricated. This executable notebook inspects provenance and preserves no-call versus reference/reference.

In [ ]:
import os, sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
from genome_evidence.notebook_support import resolve_settings
SETTINGS = resolve_settings()
PROFILE = SETTINGS.profile
REPOSITORY_URL = SETTINGS.repository_url
REPOSITORY_REF = SETTINGS.repository_ref
WORKSPACE_ROOT = SETTINGS.workspace_root
SUBJECT_ID = SETTINGS.subject_id
print({"profile": PROFILE, "requested_ref": REPOSITORY_REF})


In [ ]:
if PROFILE == "personal_drive":
    from genome_evidence.workspace import validate_workspace

    validate_workspace(WORKSPACE_ROOT)
else:
    assert PROFILE == "synthetic_ci"

In [ ]:
# ruff: noqa
import json, tempfile
from hashlib import sha256
from pathlib import Path
from genome_evidence.ingest import Ingest23andMeConfig, ingest_23andme
from genome_evidence.normalization import NormalizationConfig, normalize_m1_run

root = Path(tempfile.mkdtemp())
source = root / "synthetic.txt"
source.write_text("# genome build: GRCh38\nsynthetic_ref\t1\t5\tAA\nsynthetic_no_call\t1\t8\t--\n")
markers = root / "markers.json"
markers.write_text(
    json.dumps(
        [
            {
                "marker_id": "synthetic_ref",
                "assembly": "GRCh38",
                "chromosome": "1",
                "position": 5,
                "reference": "A",
                "alternate": "G",
                "orientation": "none",
                "orientation_authoritative": True,
            },
            {
                "marker_id": "synthetic_no_call",
                "assembly": "GRCh38",
                "chromosome": "1",
                "position": 8,
                "reference": "A",
                "alternate": "T",
                "orientation": "none",
                "orientation_authoritative": True,
            },
        ]
    )
)
fasta = root / "reference.fa"
fasta.write_text(">1\n" + "A" * 20 + "\n")
m1 = ingest_23andme(source, root / "m1", Ingest23andMeConfig(genome_build_override="GRCh38"))
m2 = normalize_m1_run(
    root / "m1",
    root / "m2",
    NormalizationConfig(marker_definitions=markers, target_reference=fasta),
)
manifest = json.loads((root / "m2/manifest.json").read_text())
assert all(
    sha256((root / "m2" / name).read_bytes()).hexdigest() == digest
    for name, digest in manifest["artifacts"].items()
)
assert len(m1.observations) == 2 and len(m2.genotypes) == 1
assert m1.observations[1].call_status.value == "no_call"
assert m2.genotypes[0].alleles == ("A", "A")
manifest["run_id"], len(m2.mappings), len(m2.genotypes)